## Step 1 — Install & Imports

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

packages = [
    "torch>=2.0.0", "transformers>=4.35.0", "scikit-learn>=1.3.0",
    "numpy>=1.24.0", "matplotlib>=3.7.0", "seaborn>=0.12.0",
    "pandas>=2.0.0", "rdkit", "tqdm",
]
for p in packages:
    install(p)
print("All packages installed")


All packages installed


In [1]:
import os, json, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModel

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("All imports OK")


Device: cuda
All imports OK


## Step 2 — Config
All hyperparameters in one place.

In [2]:
CFG = {
    # ── Paths ─────────────────────────────────────────────────
    "atc_path"          : "ATC_SMILES_SingleLabel_MultiClass.csv",
    "drugbank_path"     : "df_drugbank_smiles.csv",
    "output_dir"        : "arc_output",

    # ── Molecule filters ──────────────────────────────────────
    "min_ha"            : 5,
    "max_ha"            : 100,
    "min_mw"            : 100,
    "max_mw"            : 1500,

    # ── Data split ────────────────────────────────────────────
    "train_ratio"       : 0.70,
    "val_ratio"         : 0.15,
    "test_ratio"        : 0.15,
    "classes_per_task"  : 2,
    "min_class_samples" : 10,

    # ── MolFormer ─────────────────────────────────────────────
    "molformer_name"    : "ibm/MoLFormer-XL-both-10pct",
    "molformer_batch"   : 32,
    "max_smiles_len"    : 202,

    # ── Classifier training ───────────────────────────────────
    "clf_epochs"        : 200,
    "clf_batch"         : 32,
    "es_patience"       : 20,
    "es_min_delta"      : 1e-4,

    

    # ── Reproducibility ───────────────────────────────────────
    "seed"              : 42,

    # ── ARC shared defaults (used by sweep/BWT/ablation) ──────────────────
    "arc_lr"            : 5e-5,    # overridden per-dataset where needed
    "arc_temp"          : 2.0,

    # ══ ATC (14-class CIL) specific ══════════════════════════
    "atc_clf_lr_task0"  : 5e-4,
    "atc_clf_lr_others" : 3e-4,
    "atc_ewc_lambda"    : 500.0,
    "atc_arc_epsilon"   : 0.45,
    "atc_arc_theta"     : 0.20,
    "atc_arc_temp"      : 2.0,
    "atc_arc_lr"        : 5e-5,

    # ══ DrugBank specific ════════════════════════════════════
    "db_clf_lr_task0"   : 5e-4,
    "db_clf_lr_task1"   : 3e-4,
    "db_clf_lr_task2"   : 3e-4,
    "db_ewc_lambda"     : 8000.0,
    "db_arc_epsilon"    : 0.85,
    "db_arc_theta"      : 0.10,
    "db_arc_temp"       : 2.0,
    "db_arc_lr"         : 1e-5,
    "db_arc_temp"        : 2.0,
}
os.makedirs(CFG["output_dir"], exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print("Config ready")
print(f'  ATC      -> lr_t0={CFG["atc_clf_lr_task0"]}  lr_others={CFG["atc_clf_lr_others"]}  ewc={CFG["atc_ewc_lambda"]}')
print(f'  ATC      -> arc_eps={CFG["atc_arc_epsilon"]}  arc_theta={CFG["atc_arc_theta"]}')
print(f'  DrugBank -> lr_t0={CFG["db_clf_lr_task0"]}   lr_t1={CFG["db_clf_lr_task1"]}   ewc={CFG["db_ewc_lambda"]}')
print(f'  DrugBank -> arc_eps={CFG["db_arc_epsilon"]}   arc_theta={CFG["db_arc_theta"]}')


Config ready
  ATC      -> lr_t0=0.0005  lr_others=0.0003  ewc=500.0
  ATC      -> arc_eps=0.45  arc_theta=0.2
  DrugBank -> lr_t0=0.0005   lr_t1=0.0003   ewc=8000.0
  DrugBank -> arc_eps=0.85   arc_theta=0.1


## Step 3 — ATC Data Pipeline (14-class CIL)
14 ATC Level-1 drug classes → 7 CIL tasks × 2 classes each.

Data: `ATC_SMILES_SingleLabel_MultiClass.csv` — columns: `KEGG_Drug_ID`, `CanSmiles`, `class_index`

In [3]:
# SMILES cleaning and scaffold extraction helpers
def clean_smiles(smiles, cfg):
    if not isinstance(smiles, str) or smiles.strip() == "":
        return None, "missing_or_empty"
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, "invalid_smiles"
    try:
        mol = MolStandardize.rdMolStandardize.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    try:
        mol = MolStandardize.rdMolStandardize.Uncharger().uncharge(mol)
        mol = MolStandardize.rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    except Exception:
        pass
    try:
        ha = mol.GetNumHeavyAtoms()
        mw = rdMolDescriptors.CalcExactMolWt(mol)
        if not (cfg["min_ha"] <= ha <= cfg["max_ha"] and cfg["min_mw"] <= mw <= cfg["max_mw"]):
            return None, "filtered_out"
    except Exception:
        return None, "filter_error"
    canon = Chem.MolToSmiles(mol, canonical=True)
    return (canon, "ok") if canon else (None, "canon_failed")

def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        sc = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(sc, canonical=True)
    except Exception:
        return smiles

print("SMILES helpers defined")


SMILES helpers defined


In [4]:
# Stratified scaffold split
def stratified_scaffold_split(df, label_col, train_r, val_r, test_r, seed=42):
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in sorted(df[label_col].unique()):
        cls_df = df[df[label_col] == cls]
        sc2idx = defaultdict(list)
        for idx, sc in zip(cls_df.index, cls_df["scaffold"]):
            sc2idx[sc].append(idx)
        groups = list(sc2idx.values())
        rng.shuffle(groups)
        n    = len(cls_df)
        n_tr = max(1, int(n * train_r))
        n_vl = max(1, int(n * val_r))
        cls_tr, cls_vl, cls_ts = [], [], []
        for g in groups:
            if   len(cls_tr) < n_tr: cls_tr.extend(g)
            elif len(cls_vl) < n_vl: cls_vl.extend(g)
            else:                    cls_ts.extend(g)
        if len(cls_vl) == 0 and len(cls_tr) > 1: cls_vl.append(cls_tr.pop())
        if len(cls_ts) == 0 and len(cls_tr) > 1: cls_ts.append(cls_tr.pop())
        train_idx.extend(cls_tr); val_idx.extend(cls_vl); test_idx.extend(cls_ts)
    return df.loc[train_idx].copy(), df.loc[val_idx].copy(), df.loc[test_idx].copy()

def verify_split(train_df, val_df, test_df, label_col="label_id"):
    for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        dist = df[label_col].value_counts().sort_index().to_dict()
        print(f"  {name:5s}: {len(df):4d} samples | classes: {dist}")
    tsc = set(train_df["scaffold"]); vsc = set(val_df["scaffold"]); esc = set(test_df["scaffold"])
    print(f"  Scaffold leakage -> Train∩Val={len(tsc&vsc)} | Train∩Test={len(tsc&esc)} | Val∩Test={len(vsc&esc)}")

print("Stratified scaffold split defined")


Stratified scaffold split defined


In [5]:
# ATC label map — 14 ATC Level-1 drug classes
ATC_LABEL_MAP = {
    0:  "Alimentary_Metabolism",
    1:  "Blood_BloodForming",
    2:  "Cardiovascular",
    3:  "Dermatologicals",
    4:  "GenitoUrinary_SexHormones",
    5:  "Systemic_Hormonal",
    6:  "Antiinfectives_Systemic",
    7:  "Antineoplastic_Immunomodulating",
    8:  "MusculoSkeletal",
    9:  "Nervous_System",
    10: "Antiparasitic",
    11: "Respiratory",
    12: "Sensory_Organs",
    13: "Various",
}

# Load ATC CSV
print("=== Loading ATC dataset ===")
atc_raw = pd.read_csv(CFG["atc_path"])
print(f"  Raw shape: {atc_raw.shape}")
print(f"  Columns: {atc_raw.columns.tolist()}")
print("  Class distribution:")
print(atc_raw["class_index"].value_counts().sort_index())

# Clean SMILES
res_atc = atc_raw["CanSmiles"].apply(lambda s: clean_smiles(s, CFG))
atc_raw["canon_smiles"] = res_atc.apply(lambda x: x[0])
atc_raw["clean_status"] = res_atc.apply(lambda x: x[1])
atc_df = atc_raw[atc_raw["clean_status"] == "ok"].drop_duplicates("canon_smiles").copy()
print(f"\n  After cleaning: {atc_df.shape}")

atc_df["label_id"] = atc_df["class_index"].astype(int)
atc_df["scaffold"] = atc_df["canon_smiles"].apply(get_scaffold)

print(f"Final ATC dataset: {atc_df.shape}")
print("Class distribution after cleaning:")
print(atc_df["label_id"].value_counts().sort_index())


=== Loading ATC dataset ===
  Raw shape: (3921, 3)
  Columns: ['KEGG_Drug_ID', 'CanSmiles', 'class_index']
  Class distribution:
class_index
0     427
1     108
2     496
3     145
4     170
5      65
6     487
7     384
8     183
9     773
10    118
11    268
12     88
13    209
Name: count, dtype: int64

  After cleaning: (2534, 5)
Final ATC dataset: (2534, 7)
Class distribution after cleaning:
label_id
0     275
1      68
2     331
3     109
4     131
5      42
6     302
7     258
8     129
9     474
10     86
11    157
12     60
13    112
Name: count, dtype: int64


In [6]:
# Stratified scaffold split for ATC
atc_train, atc_val, atc_test = stratified_scaffold_split(
    atc_df, "label_id",
    CFG["train_ratio"], CFG["val_ratio"], CFG["test_ratio"], CFG["seed"]
)
print(f"ATC split: train={len(atc_train)} | val={len(atc_val)} | test={len(atc_test)}")
verify_split(atc_train, atc_val, atc_test)


ATC split: train=1771 | val=415 | test=348
  Train: 1771 samples | classes: {0: 192, 1: 47, 2: 231, 3: 76, 4: 91, 5: 30, 6: 211, 7: 181, 8: 92, 9: 331, 10: 60, 11: 109, 12: 42, 13: 78}
  Val  :  415 samples | classes: {0: 41, 1: 10, 2: 54, 3: 16, 4: 21, 5: 6, 6: 45, 7: 40, 8: 30, 9: 72, 10: 13, 11: 24, 12: 10, 13: 33}
  Test :  348 samples | classes: {0: 42, 1: 11, 2: 46, 3: 17, 4: 19, 5: 6, 6: 46, 7: 37, 8: 7, 9: 71, 10: 13, 11: 24, 12: 8, 13: 1}
  Scaffold leakage -> Train∩Val=34 | Train∩Test=32 | Val∩Test=18


In [26]:
from sklearn.utils import resample

def oversample_minority(df, label_col, min_count, seed=42):
    dfs = []
    for cls in df[label_col].unique():
        cls_df = df[df[label_col] == cls]
        if len(cls_df) < min_count:
            cls_df = resample(cls_df, replace=True, n_samples=min_count, random_state=seed)
        dfs.append(cls_df)
    return pd.concat(dfs, ignore_index=True).sample(frac=1, random_state=seed)

# Oversample minority classes to at least 50 samples in train
atc_train_bal = oversample_minority(atc_train, "label_id", min_count=50)
print(f"ATC balanced train: {len(atc_train_bal)} samples")
print(atc_train_bal["label_id"].value_counts().sort_index())

# Build CIL task list — 2 new classes per task -> 7 tasks
def make_cil_tasks(train_df, val_df, test_df, cpt, label_map):
    all_ids = sorted(train_df["label_id"].unique())
    tasks = []
    for t in range(int(np.ceil(len(all_ids) / cpt))):
        new_cls  = all_ids[t*cpt : (t+1)*cpt]
        seen_cls = all_ids[: (t+1)*cpt]
        tasks.append({
            "task_id"    : t,
            "new_classes": new_cls,
            "all_classes": list(seen_cls),
            "n_classes"  : len(seen_cls),
            "class_names": {lid: label_map[lid] for lid in seen_cls},
        })
    return tasks

atc_tasks = make_cil_tasks(atc_train_bal, atc_val, atc_test, CFG["classes_per_task"], ATC_LABEL_MAP)
print(f"\nATC CIL tasks: {len(atc_tasks)}")
for t in atc_tasks:
    print(f'  Task {t["task_id"]}: new={[t["class_names"][c] for c in t["new_classes"]]}  total_classes={t["n_classes"]}')


ATC balanced train: 1802 samples
label_id
0     192
1      50
2     231
3      76
4      91
5      50
6     211
7     181
8      92
9     331
10     60
11    109
12     50
13     78
Name: count, dtype: int64

ATC CIL tasks: 7
  Task 0: new=['Alimentary_Metabolism', 'Blood_BloodForming']  total_classes=2
  Task 1: new=['Cardiovascular', 'Dermatologicals']  total_classes=4
  Task 2: new=['GenitoUrinary_SexHormones', 'Systemic_Hormonal']  total_classes=6
  Task 3: new=['Antiinfectives_Systemic', 'Antineoplastic_Immunomodulating']  total_classes=8
  Task 4: new=['MusculoSkeletal', 'Nervous_System']  total_classes=10
  Task 5: new=['Antiparasitic', 'Respiratory']  total_classes=12
  Task 6: new=['Sensory_Organs', 'Various']  total_classes=14


## Step 4 — DrugBank Data Pipeline
DrugBank multi-label → primary label assignment → stratified scaffold split.

In [28]:
db_raw = pd.read_csv(CFG["drugbank_path"])
print(f"DrugBank raw: {db_raw.shape}")

smiles_col_db = "smiles" if "smiles" in db_raw.columns else "SMILES"
res_db = db_raw[smiles_col_db].apply(lambda s: clean_smiles(s, CFG))
db_raw["canon_smiles"] = res_db.apply(lambda x: x[0])
db_raw["clean_status"] = res_db.apply(lambda x: x[1])
db_df = db_raw[db_raw["clean_status"] == "ok"].drop_duplicates("canon_smiles").copy()
print(f"After cleaning: {db_df.shape}")

IGNORE_COLS = {
    "Unnamed: 0","drugbank_id","name","cas","smiles","SMILES",
    "canon_smiles","clean_status","logP ALOGPS","logP ChemAxon",
    "solubility ALOGPS","pKa (strongest acidic)","pKa (strongest basic)","F"
}
DB_LABEL_COLS = [c for c in db_df.columns if c not in IGNORE_COLS]

def assign_db_label(row):
    vals = row[DB_LABEL_COLS].fillna(0)
    return "none_detected" if vals.sum() == 0 else vals.idxmax()

db_df["primary_label"] = db_df.apply(assign_db_label, axis=1)
label_counts = db_df["primary_label"].value_counts()
valid_labels = label_counts[label_counts >= CFG["min_class_samples"]].index.tolist()
db_df        = db_df[db_df["primary_label"].isin(valid_labels)].copy()

le_db = LabelEncoder()
db_df["label_id"] = le_db.fit_transform(db_df["primary_label"])
db_label_map      = {int(e): c for e, c in enumerate(le_db.classes_)}
db_df["scaffold"] = db_df["canon_smiles"].apply(get_scaffold)

db_train, db_val, db_test = stratified_scaffold_split(
    db_df, "label_id", CFG["train_ratio"], CFG["val_ratio"], CFG["test_ratio"], CFG["seed"])
print(f"DrugBank: {db_df.shape}  ->  train={len(db_train)} val={len(db_val)} test={len(db_test)}")
verify_split(db_train, db_val, db_test)

db_tasks = make_cil_tasks(db_train, db_val, db_test, CFG["classes_per_task"], db_label_map)
print(f"\nDrugBank CIL tasks: {len(db_tasks)}")
for t in db_tasks:
    print(f'  Task {t["task_id"]}: new={[t["class_names"][c] for c in t["new_classes"]]}')


DrugBank raw: (5653, 16)
After cleaning: (5361, 18)
DrugBank: (5360, 21)  ->  train=3761 val=804 test=795
  Train: 3761 samples | classes: {0: 2673, 1: 37, 2: 894, 3: 15, 4: 142}
  Val  :  804 samples | classes: {0: 572, 1: 6, 2: 190, 3: 4, 4: 32}
  Test :  795 samples | classes: {0: 574, 1: 1, 2: 189, 3: 2, 4: 29}
  Scaffold leakage -> Train∩Val=22 | Train∩Test=30 | Val∩Test=13

DrugBank CIL tasks: 3
  Task 0: new=['carbonyl', 'nitro']
  Task 1: new=['none_detected', 'sulfinyl']
  Task 2: new=['sulfonyl']


## Step 5 — MolFormer Feature Extraction
Frozen MolFormer backbone — mean-pooled embeddings used as molecular features.
No fine-tuning is performed.

In [ ]:
print("Loading MolFormer...")
tokenizer = AutoTokenizer.from_pretrained(CFG["molformer_name"], trust_remote_code=True)
molformer = AutoModel.from_pretrained(CFG["molformer_name"], trust_remote_code=True, deterministic_eval=True)
molformer.eval().to(DEVICE)
FEAT_DIM = molformer.config.hidden_size
for p in molformer.parameters():
    p.requires_grad = False
print(f"MolFormer loaded  |  hidden_dim={FEAT_DIM}  (frozen)")


Loading MolFormer...
MolFormer loaded  |  hidden_dim=768  (frozen)


In [ ]:
@torch.no_grad()
def extract_molformer_features(smiles_list, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc="MolFormer encode"):
        batch  = smiles_list[i: i + batch_size]
        enc    = tokenizer(batch, padding=True, truncation=True,
                           max_length=CFG["max_smiles_len"], return_tensors="pt").to(DEVICE)
        out    = molformer(**enc)
        hidden = out.last_hidden_state
        mask_f = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        all_emb.append(pooled.cpu().numpy())
    return np.vstack(all_emb)

def get_feats(split_df, idx_to_feat):
    X = np.stack([idx_to_feat[i] for i in split_df.index])
    y = split_df["label_id"].values
    return X, y

print("Feature extractor ready")


Feature extractor ready


In [ ]:
# Feature extraction for ATC
print("=== ATC feature extraction ===")
atc_feats_all   = extract_molformer_features(atc_df["canon_smiles"].tolist(), CFG["molformer_batch"])
idx_to_feat_atc = {idx: feat for idx, feat in zip(atc_df.index, atc_feats_all)}

atc_X_val, atc_y_val = get_feats(atc_val,  idx_to_feat_atc)
atc_X_te,  atc_y_te  = get_feats(atc_test, idx_to_feat_atc)

atc_X_tr = extract_molformer_features(atc_train_bal["canon_smiles"].tolist(), CFG["molformer_batch"])
atc_y_tr = atc_train_bal["label_id"].values

print(f"ATC -> train:{atc_X_tr.shape} | val:{atc_X_val.shape} | test:{atc_X_te.shape}")

# Feature extraction for DrugBank
print("\n=== DrugBank feature extraction ===")
db_feats_all   = extract_molformer_features(db_df["canon_smiles"].tolist(), CFG["molformer_batch"])
idx_to_feat_db = {idx: feat for idx, feat in zip(db_df.index, db_feats_all)}
db_X_tr,  db_y_tr  = get_feats(db_train, idx_to_feat_db)
db_X_val, db_y_val = get_feats(db_val,   idx_to_feat_db)
db_X_te,  db_y_te  = get_feats(db_test,  idx_to_feat_db)
print(f"DrugBank -> train:{db_X_tr.shape} | val:{db_X_val.shape} | test:{db_X_te.shape}")


=== ATC feature extraction ===


MolFormer encode:   0%|          | 0/80 [00:00<?, ?it/s]

MolFormer encode:   0%|          | 0/57 [00:00<?, ?it/s]

ATC -> train:(1802, 768) | val:(415, 768) | test:(348, 768)

=== DrugBank feature extraction ===


MolFormer encode:   0%|          | 0/168 [00:00<?, ?it/s]

DrugBank -> train:(3761, 768) | val:(804, 768) | test:(795, 768)


## Step 5b — Feature Separability Diagnostic
Logistic Regression probe to verify MolFormer embeddings are separable before CIL.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

print("=" * 65)
print("  ATC FEATURE SEPARABILITY DIAGNOSTIC")
print("=" * 65)

# Per-task pairwise probe (first 4 tasks)
for t_idx in range(min(4, len(atc_tasks))):
    task    = atc_tasks[t_idx]
    new_cls = task["new_classes"]
    tr_mask = np.isin(atc_y_tr, new_cls)
    te_mask = np.isin(atc_y_te, new_cls)
    X_tr_t, y_tr_t = atc_X_tr[tr_mask], atc_y_tr[tr_mask]
    X_te_t, y_te_t = atc_X_te[te_mask], atc_y_te[te_mask]
    if len(np.unique(y_tr_t)) < 2 or len(X_te_t) == 0:
        print(f"  Task {t_idx}: skipped (insufficient data)")
        continue
    lr = LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)
    lr.fit(X_tr_t, y_tr_t)
    pred = lr.predict(X_te_t)
    f1   = f1_score(y_te_t, pred, average="macro", zero_division=0)
    try:
        prob  = lr.predict_proba(X_te_t)
        auroc = roc_auc_score(
            y_te_t,
            prob[:, 1] if prob.shape[1] == 2 else prob,
            multi_class="ovr", average="macro"
        )
    except Exception:
        auroc = float("nan")
    names    = [ATC_LABEL_MAP[c] for c in new_cls]
    f1_ok    = "OK" if f1    >= 0.55 else "WEAK"
    auroc_ok = "OK" if auroc >= 0.60 else "WEAK"
    print(f"  Task {t_idx} {names}: F1={f1:.4f}[{f1_ok}]  AUROC={auroc:.4f}[{auroc_ok}]")

# Combined 14-class probe
print("\n-- Combined 14-class ATC --")
lr_all = LogisticRegression(max_iter=1000, class_weight="balanced",
                            multi_class="multinomial", random_state=42)
lr_all.fit(atc_X_tr, atc_y_tr)
pred_all  = lr_all.predict(atc_X_te)
probs_all = lr_all.predict_proba(atc_X_te)
f1_all    = f1_score(atc_y_te, pred_all, average="macro", zero_division=0)
try:
    auroc_all = roc_auc_score(atc_y_te, probs_all, multi_class="ovr", average="macro")
except Exception as e:
    auroc_all = float("nan")
    print(f"  AUROC error: {e}")
print(f"  14-class Macro-F1 : {f1_all:.4f}")
print(f"  14-class AUROC    : {auroc_all:.4f}")
print("\n" + "=" * 65)
if f1_all >= 0.40:
    print("  PASS: Features separable -- proceed with CIL pipeline")
else:
    print("  WARN: Features may be weak for some classes -- EWC/ARC tuning is important")
print("=" * 65)


  ATC FEATURE SEPARABILITY DIAGNOSTIC
  Task 0 ['Alimentary_Metabolism', 'Blood_BloodForming']: F1=0.7307[OK]  AUROC=0.8247[OK]
  Task 1 ['Cardiovascular', 'Dermatologicals']: F1=0.7583[OK]  AUROC=0.8875[OK]
  Task 2 ['GenitoUrinary_SexHormones', 'Systemic_Hormonal']: F1=0.6250[OK]  AUROC=0.6404[OK]
  Task 3 ['Antiinfectives_Systemic', 'Antineoplastic_Immunomodulating']: F1=0.7372[OK]  AUROC=0.7996[OK]

-- Combined 14-class ATC --
  14-class Macro-F1 : 0.2828
  14-class AUROC    : 0.7594

  WARN: Features may be weak for some classes -- EWC/ARC tuning is important


## Step 6 — CIL Classifier + EWC
Expanding linear head grows by 2 nodes per task. EWC protects past weights.

In [13]:
class MolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ExpandingLinearHead(nn.Module):
    def __init__(self, in_dim, n_init):
        super().__init__()
        self.hidden = nn.Linear(in_dim, 512)
        self.ln     = nn.LayerNorm(512)
        self.relu   = nn.ReLU()
        self.drop   = nn.Dropout(0.2)
        self.fc     = nn.Linear(512, n_init)

    def grow(self, n_new, device):
        old    = self.fc
        n_old  = old.out_features
        new_fc = nn.Linear(old.in_features, n_old + n_new)
        with torch.no_grad():
            new_fc.weight[:n_old] = old.weight
            new_fc.bias[:n_old]   = old.bias
            nn.init.xavier_uniform_(new_fc.weight[n_old:])
            nn.init.zeros_(new_fc.bias[n_old:])
        self.fc = new_fc.to(device)

    def forward(self, x):
        return self.fc(self.drop(self.relu(self.ln(self.hidden(x)))))

    @property
    def n_classes(self): return self.fc.out_features


class EWC:
    def __init__(self, model, dataloader, device, n_classes):
        self.params = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}
        self.fisher = self._compute_fisher(model, dataloader, device, n_classes)

    def _compute_fisher(self, model, dataloader, device, n_classes):
        fisher    = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
        model.eval()
        n_samples = 0
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            model.zero_grad()
            F.cross_entropy(model(xb)[:, :n_classes], yb).backward()
            for n, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.detach().pow(2) * len(xb)
            n_samples += len(xb)
        for n in fisher:
            fisher[n] /= max(n_samples, 1)
        return fisher

    def penalty(self, model):
        loss = 0.0
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.fisher:
                slices = tuple(slice(0, s) for s in self.fisher[n].shape)
                loss += (self.fisher[n] * (p[slices] - self.params[n]).pow(2)).sum()
        return loss


print("ExpandingLinearHead + EWC defined")


ExpandingLinearHead + EWC defined


In [14]:
def compute_macro_f1(clf, X_v, y_v, device):
    if len(X_v) == 0: return float("nan")
    clf.eval()
    with torch.no_grad():
        pred = clf(torch.tensor(X_v, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
    return float(f1_score(y_v, pred, labels=np.unique(y_v), average="macro", zero_division=0))


def train_classifier_on_task(clf, X_tr, y_tr, X_val, y_val,
                              seen_cls, epochs, lr, batch_size, device,
                              es_patience=10, es_min_delta=1e-4,
                              ewc=None, ewc_lambda=400.0):
    tr_mask  = np.isin(y_tr, seen_cls)
    X_t, y_t = X_tr[tr_mask], y_tr[tr_mask]
    X_v, y_v = X_val[np.isin(y_val, seen_cls)], y_val[np.isin(y_val, seen_cls)]

    weight_tensor = torch.ones(clf.n_classes, dtype=torch.float32).to(device)
    seen_counts   = np.array([max(1, (y_t == c).sum()) for c in seen_cls])
    seen_weights  = 1.0 / seen_counts.astype(float)
    seen_weights  = seen_weights / seen_weights.sum() * len(seen_cls)
    for idx, c in enumerate(seen_cls):
        weight_tensor[c] = float(seen_weights[idx])

    dl      = DataLoader(MolDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    opt     = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

    best_val_f1, best_weights, patience_count = -1.0, copy.deepcopy(clf.state_dict()), 0
    clf.train()
    for ep in range(epochs):
        ep_loss = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            ewc_loss = ewc.penalty(clf) if ewc is not None else 0.0
            (loss_fn(clf(xb), yb) + ewc_lambda * ewc_loss).backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            opt.step()
            ep_loss += loss_fn(clf(xb.detach()), yb.detach()).item()

        val_f1 = compute_macro_f1(clf, X_v, y_v, device)
        if not np.isnan(val_f1) and val_f1 - best_val_f1 >= es_min_delta:
            best_val_f1, best_weights, patience_count = val_f1, copy.deepcopy(clf.state_dict()), 0
        else:
            patience_count += 1
        if (ep + 1) % 10 == 0:
            ewc_str = f"  ewc={float(ewc_loss):.5f}" if ewc is not None else ""
            print(f"  Epoch {ep+1:3d}/{epochs}  loss={ep_loss/max(len(dl),1):.4f}"
                  f"  val_f1={val_f1:.3f}{ewc_str}  patience={patience_count}/{es_patience}")
        if patience_count >= es_patience:
            print(f"  [Early Stop] epoch {ep+1}  best_val_f1={best_val_f1:.4f}")
            break
        clf.train()

    clf.load_state_dict(best_weights)
    clf.eval()
    return clf


print("train_classifier_on_task defined")


train_classifier_on_task defined


In [15]:
def compute_batch_theta(logits, task_id, n_classes_per_task, base_theta):
    """
    Computes a dynamic theta (threshold multiplier) for each sample in a batch.
    """
    # Example logic: if you just want to pass the static scalar theta across the batch
    # as a tensor matching the batch size:
    batch_size = logits.shape[0]
    device = logits.device
    
    # Simple broadcasted base threshold
    batch_theta = torch.full((batch_size,), base_theta, device=device)
    
    # Optional: Insert adaptive calculation logic here if needed
    # e.g., adjusting theta based on standard deviation or max logit values
    
    return batch_theta

## Step 7 — Out-of-Task Detection (OTD)
Paper Algorithm 1. Assumption 1 → retention. Assumption 2 → correction.

In [16]:
def out_of_task_detection(logits, task_id, n_classes_per_task, epsilon, theta, batch_theta=None):
    s = n_classes_per_task
    past_boundary = s * task_id
    probs = F.softmax(logits.float(), dim=-1)
    pred_cls = probs.argmax(dim=-1)

    decisions = []
    for i in range(logits.shape[0]):
        # If we are on the first task, everything is considered "current"
        if past_boundary == 0:
            decisions.append("current")
            continue

        c = probs[i].max().item()      # Global maximum probability (confidence)
        y_hat = pred_cls[i].item()     # Global predicted class

        # ==========================================
        # 🚨 OVERCONFIDENCE GATE (e.g., for DrugBank) 🚨
        # If the model is already >95% confident in its prediction,
        # skip ARC entirely (NO retention, NO correction). 
        # Leave the prediction as it is to prevent degrading perfect weights.
        # ==========================================
        if c > 0.95:
            decisions.append("current")
            continue

        # ==========================================
        # ARC Core Logic Below
        # ==========================================
        
        # Assumption 1: The predicted class belongs to a PAST task
        if y_hat < past_boundary: 
            # If confidence is above the threshold (epsilon), perform Retention
            if c >= epsilon:
                decisions.append("retention")
            else:
                decisions.append("current")
                
        # Assumption 2: The predicted class belongs to the CURRENT task
        else: 
            # Calculate the maximum probability restricted ONLY to past tasks
            past_slice_probs = F.softmax(logits[i, :past_boundary].float(), dim=-1)
            c_hat = past_slice_probs.max().item()
            
            # Ratio of (Highest Past Confidence / Highest Overall Confidence)
            w_inverted = c_hat / (c + 1e-8) 
            
            # If the past class is fighting closely with the current class 
            # (e.g., past confidence is > 85% of current confidence when theta=0.15),
            # the model is confused. Apply Correction.
            if w_inverted > (1.0 - theta):
                decisions.append("correction")
            else:
                decisions.append("current")

    return decisions, probs, pred_cls

## Step 8 — Adaptive Retention (Paper Eq. 2 & 3)
L = L_CE(pseudo_label) + L_EM. One SGD step per retained sample.

In [17]:
def adaptive_retention_step(clf, x_single, pseudo_label, lr, device):
    """
    Adaptive Retention — Fixed for High-Accuracy Datasets (Like DrugBank).
    """
    clf.eval()  # First check logits in eval mode
    with torch.no_grad():
        x = x_single.detach().clone().to(device)
        logits = clf(x)
        probs = F.softmax(logits, dim=-1)
        
        # EARLY EXIT: If model is already >95% confident, skip retention to preserve weights
        if probs.max().item() > 0.95:
            return 

    # Only run retention training if confidence < 95%
    clf.train()
    for p in clf.parameters():
        p.requires_grad_(True)

    opt = torch.optim.SGD(clf.parameters(), lr=lr)
    opt.zero_grad()
    clf.zero_grad()

    logits_train = clf(x)
    label_t = torch.tensor([pseudo_label], dtype=torch.long).to(device)

    l_ce  = F.cross_entropy(logits_train, label_t)
    probs_train = F.softmax(logits_train, dim=-1)
    l_em  = -(probs_train * probs_train.clamp(min=1e-8).log()).sum(dim=-1).mean()
    
    (l_ce + l_em).backward()

    torch.nn.utils.clip_grad_norm_(clf.parameters(), max_norm=0.1)
    opt.step()
    clf.eval()

## Step 9 — Adaptive Correction: Task-based Softmax Score (TSS)
Paper Definition 1.

In [18]:
def compute_tss(logits_single, task_id, n_classes_per_task, temperature):
    """
    Robust Adaptive Correction: 
    Masks out the current/future task classes and securely picks the highest 
    probability class from the seen past tasks.
    """
    past_boundary = task_id * n_classes_per_task
    
    # Isolate only the logits belonging to old tasks
    past_logits = logits_single[:past_boundary]
    
    # Pick the absolute best class from the old tasks
    best_past_cls = past_logits.argmax().item()
    
    return best_past_cls

## Step 10 — Evaluation Functions
AUROC uses post-ARC probs. OTD counts printed per task.

In [19]:
def compute_auroc(y_t, probs_np, seen_cls, n_total_cls):
    valid_seen = [c for c in seen_cls if c < n_total_cls]
    if len(valid_seen) < 2: return float("nan")
    y_t_arr    = np.asarray(y_t)
    y_bin      = np.stack([(y_t_arr == c).astype(int) for c in valid_seen], axis=1)
    probs_seen = probs_np[:, valid_seen]
    auroc_list = []
    valid_count = 0
    for col_i in range(len(valid_seen)):
        col_true = y_bin[:, col_i]
        if col_true.sum() == 0 or col_true.sum() == len(col_true): continue
        valid_count += 1
        try: auroc_list.append(roc_auc_score(col_true, probs_seen[:, col_i]))
        except Exception: pass
    if valid_count > 0 and len(auroc_list) < valid_count:
        print(f"    [AUROC warn] {valid_count - len(auroc_list)}/{valid_count} classes skipped (single-label)")
    return float(np.mean(auroc_list)) if auroc_list else float("nan")


def evaluate_task(clf, X_test, y_test, seen_cls, task_id=None, n_cpt=None,
                   epsilon=None, theta=None, temp=None, arc_lr=None,
                   use_arc=False, arc_mode="full", device="cpu", persistent_clf_arc=None):
    mask     = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {"accuracy":0.0,"macro_f1":0.0,"auroc":float("nan"),
                "otd_retention":0,"otd_correction":0,"otd_current":0}, persistent_clf_arc

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all        = clf(Xt)
        baseline_probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    if use_arc and task_id is not None and task_id > 0:
        clf_arc = persistent_clf_arc if persistent_clf_arc is not None else copy.deepcopy(clf)
        with torch.no_grad():
            clf_arc.eval()
            bt = compute_batch_theta(clf_arc(Xt), task_id, n_cpt, theta)
        otd_counts = {"retention": 0, "correction": 0, "current": 0}
        preds      = []
        for i in range(len(Xt)):
            xi = torch.tensor(X_t[i:i+1], dtype=torch.float32).to(device)
            with torch.no_grad():
                clf_arc.eval()
                li = clf_arc(xi)
            dec, _, pred_i = out_of_task_detection(li, task_id, n_cpt, epsilon, theta, bt)
            d = dec[0]; pl = pred_i[0].item(); otd_counts[d] += 1

            if d == "retention" and arc_mode in ("full", "retention"):
                with torch.no_grad():
                    clf_arc.eval()
                    li_tmp     = clf_arc(xi)
                    past_probs = F.softmax(li_tmp, dim=-1)[0, :task_id * n_cpt]
                    pl         = past_probs.argmax().item()
                adaptive_retention_step(clf_arc, xi, pl, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li = clf_arc(xi)
                preds.append(F.softmax(li, dim=-1)[0, :task_id * n_cpt].argmax().item())
            elif d == "correction" and arc_mode in ("full", "correction"):
                with torch.no_grad():
                    clf_arc.eval()
                    li = clf_arc(xi)
                preds.append(compute_tss(li[0], task_id, n_cpt, temp))
            else:
                preds.append(pl)

        preds = np.array(preds)
        n = len(Xt)
        print(f"    OTD [{arc_mode}]: "
              f"retention={otd_counts['retention']}/{n} ({100*otd_counts['retention']/n:.1f}%)  "
              f"correction={otd_counts['correction']}/{n} ({100*otd_counts['correction']/n:.1f}%)  "
              f"current={otd_counts['current']}/{n} ({100*otd_counts['current']/n:.1f}%)  "
              f"batch_theta={bt[0].item():.3f}")
        with torch.no_grad():
            clf_arc.eval()
            arc_probs_np = F.softmax(clf_arc(Xt), dim=-1).cpu().numpy()
    else:
        clf_arc      = persistent_clf_arc
        otd_counts   = {"retention": 0, "correction": 0, "current": 0}
        with torch.no_grad():
            preds = logits_all.argmax(1).cpu().numpy()
        arc_probs_np = baseline_probs_np

    present  = np.unique(y_t)
    accuracy = float((preds == y_t).mean())
    macro_f1 = float(f1_score(y_t, preds, labels=present, average="macro", zero_division=0))
    auroc    = compute_auroc(y_t, arc_probs_np, seen_cls, logits_all.shape[1])
    return {"accuracy": accuracy, "macro_f1": macro_f1, "auroc": auroc,
            "otd_retention": otd_counts["retention"], "otd_correction": otd_counts["correction"],
            "otd_current": otd_counts["current"]}, clf_arc


print("evaluate_task defined")


evaluate_task defined


In [20]:
# STEP 11 — Hyperparameter Sweep for ε and θ  (Paper Fig 5)
# Grid search on validation set — finds best epsilon/theta per dataset
# ─────────────────────────────────────────────────────────────────────────────
 
def sweep_arc_thresholds(tasks, X_tr, y_tr, X_val, y_val,
                          feat_dim, cfg, device,
                          eps_values=None, theta_values=None):
    """
    Grid-search epsilon × theta on validation data.
    Returns best (epsilon, theta) and full sweep DataFrame.
    """
    if eps_values   is None: eps_values   = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
    if theta_values is None: theta_values = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
 
    # Train a shared classifier once (baseline, no ARC) for sweep
    clf_sw = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
    ewc_sw = None
    for task in tasks:
        tid = task["task_id"]
        if tid > 0:
            clf_sw.grow(len(task["new_classes"]), device)
        clf_sw = train_classifier_on_task(
            clf_sw, X_tr, y_tr, X_val, y_val, task["all_classes"],
            cfg["clf_epochs"], cfg.get("clf_lr", 1e-3), cfg["clf_batch"], device,
            es_patience=cfg.get("es_patience", 10),
            es_min_delta=cfg.get("es_min_delta", 1e-4),
            ewc=ewc_sw, ewc_lambda=cfg.get("atc_ewc_lambda", 500.0),
        )
        if tid < len(tasks) - 1:
            tr_mask = np.isin(y_tr, task["all_classes"])
            ewc_sw  = EWC(clf_sw,
                          DataLoader(MolDataset(X_tr[tr_mask], y_tr[tr_mask]),
                                     batch_size=cfg["clf_batch"], shuffle=False),
                          device, len(task["all_classes"]))
 
    rows = []
    best_f1, best_eps, best_theta = -1.0, eps_values[0], theta_values[0]
 
    for eps in eps_values:
        for tht in theta_values:
            acc_list, f1_list = [], []
            for task in tasks[1:]:   # ARC only relevant from task 1 onwards
                res, _ = evaluate_task(
                    clf_sw, X_val, y_val, task["all_classes"],
                    task_id=task["task_id"],
                    n_cpt=cfg["classes_per_task"],
                    epsilon=eps, theta=tht,
                    temp=cfg.get("arc_temp", 2.0), arc_lr=cfg.get("arc_lr", 5e-5),
                    use_arc=True, arc_mode="full", device=device,
                    persistent_clf_arc=copy.deepcopy(clf_sw),
                )
                acc_list.append(res["accuracy"])
                f1_list.append(res["macro_f1"])
            avg_f1 = float(np.mean(f1_list)) if f1_list else 0.0
            avg_ab = float(np.mean(acc_list)) if acc_list else 0.0
            rows.append({"epsilon": eps, "theta": tht, "val_F1": round(avg_f1, 4), "val_AB": round(avg_ab, 4)})
            if avg_f1 > best_f1:
                best_f1, best_eps, best_theta = avg_f1, eps, tht
 
    sweep_df = pd.DataFrame(rows).sort_values("val_F1", ascending=False)
    print(f"  Best -> epsilon={best_eps}  theta={best_theta}  val_F1={best_f1:.4f}")
    print(sweep_df.head(10).to_string(index=False))
    return best_eps, best_theta, sweep_df
 
 
print("sweep_arc_thresholds defined")
 
 


sweep_arc_thresholds defined


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 12 — Full CIL Pipeline
# Memory-free. EWC after each task. ARC at test-time only.
# Fresh arc copy per prev_task evaluation.
# ─────────────────────────────────────────────────────────────────────────────

def run_cil_pipeline(tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
                      feat_dim, label_map, cfg, device, use_arc=False,
                      arc_mode="full", name="Dataset",
                      arc_epsilon=None, arc_theta=None,
                      arc_temp=None, arc_lr=None, # <--- Ye add karna zaroori hai
                      clf_lr_per_task=None, ewc_lambda=None):
    
    print(f'\n{"="*60}\n  {name}  |  ARC={use_arc}\n{"="*60}')

    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg["seed"])

    # Params extract logic
    epsilon = arc_epsilon if arc_epsilon is not None else cfg.get("atc_arc_epsilon", 0.45)
    theta   = arc_theta   if arc_theta   is not None else cfg.get("atc_arc_theta",   0.20)
    temp    = arc_temp    if arc_temp    is not None else cfg.get("atc_arc_temp",    2.0)
    alr     = arc_lr      if arc_lr      is not None else cfg.get("atc_arc_lr",      5e-5)
    lam     = ewc_lambda  if ewc_lambda  is not None else cfg.get("atc_ewc_lambda", 500.0)

    clf             = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
    ewc_state       = None
    clf_checkpoints = {}
    R               = defaultdict(dict)
    peak_acc        = {}
    rows            = []

    for task in tasks:
        tid      = task["task_id"]
        seen_cls = task["all_classes"]
        print(f'\n-- Task {tid} | new: {[task["class_names"][c] for c in task["new_classes"]]} --')
        
        if tid > 0:
            clf.grow(len(task["new_classes"]), device)

        if clf_lr_per_task is not None and tid < len(clf_lr_per_task):
            lr_this_task = clf_lr_per_task[tid]
        else:
            lr_this_task = cfg.get("clf_lr", 1e-3)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val, seen_cls,
            cfg["clf_epochs"], lr_this_task, cfg["clf_batch"], device,
            es_patience=cfg.get("es_patience", 20),
            es_min_delta=cfg.get("es_min_delta", 1e-4),
            ewc=ewc_state, ewc_lambda=lam,
        )

        if tid < len(tasks) - 1:
            tr_mask   = np.isin(y_tr, seen_cls)
            ewc_state = EWC(clf, DataLoader(MolDataset(X_tr[tr_mask], y_tr[tr_mask]), 
                           batch_size=cfg["clf_batch"], shuffle=False), device, len(seen_cls))
            print(f"  [EWC] Fisher updated for task {tid}")

        clf_checkpoints[tid] = copy.deepcopy(clf.state_dict())

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task["task_id"]
            fresh_arc = copy.deepcopy(clf) if use_arc else None
            res, _ = evaluate_task(
                clf, X_te, y_te, prev_task["all_classes"],
                task_id=prev_tid, n_cpt=cfg["classes_per_task"],
                epsilon=epsilon, theta=theta, temp=temp, arc_lr=alr,
                use_arc=use_arc, arc_mode=arc_mode, device=device,
                persistent_clf_arc=fresh_arc,
            )
            R[tid][prev_tid] = res
            if prev_tid == tid: peak_acc[tid] = res["accuracy"]
            print(f"  Eval Task {prev_tid} -> acc={res['accuracy']:.4f}  f1={res['macro_f1']:.4f}")

    # Metrics calculation logic
    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, f1_list, auroc_list, forget_list = [], [], [], []
    total_ret = total_cor = total_cur = 0

    for prev_tid in range(n_tasks):
        r = R[final_tid][prev_tid]
        acc_list.append(r["accuracy"])
        f1_list.append(r["macro_f1"])
        auroc_list.append(r["auroc"])
        total_ret += r["otd_retention"]
        total_cor += r["otd_correction"]
        total_cur += r["otd_current"]
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - r["accuracy"])
        
        rows.append({
            "task_id"        : prev_tid,
            "class_names"    : str(list(tasks[prev_tid]["class_names"].values())),
            "final_acc"      : round(r["accuracy"], 4),
            "final_macro_f1" : round(r["macro_f1"], 4),
            "final_auroc"    : round(r["auroc"], 4) if not np.isnan(r["auroc"]) else float("nan"),
            "peak_acc"       : round(peak_acc.get(prev_tid, r["accuracy"]), 4),
            "forgetting"     : round(peak_acc.get(prev_tid, r["accuracy"]) - r["accuracy"], 4),
        })

    avg_acc    = float(np.mean(acc_list))
    avg_f1     = float(np.mean(f1_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*55}')
    print(f"  Average Accuracy (AB) : {avg_acc:.4f}")
    print(f"  Average Macro-F1      : {avg_f1:.4f}")
    print(f"  Forgetting (F)        : {forgetting:.4f}")
    print(f"  Average AUROC         : {avg_auroc:.4f}")
    if use_arc:
        total = max(total_ret + total_cor + total_cur, 1)
        print(f"  OTD: retention={total_ret}({100*total_ret/total:.1f}%)  "
              f"correction={total_cor}({100*total_cor/total:.1f}%)  "
              f"current={total_cur}({100*total_cur/total:.1f}%)")
    print(f'{"─"*55}')

    return pd.DataFrame(rows), avg_acc, avg_f1, forgetting, avg_auroc, clf_checkpoints

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 13 — BWT / FWT + OTD Stats
# Backward Transfer = avg forgetting. Forward Transfer = avg improvement
# on future tasks before seeing them.
# ─────────────────────────────────────────────────────────────────────────────
 
def compute_bwt_fwt_from_checkpoints(tasks, X_te, y_te, feat_dim, clf_checkpoints,
                                      cfg, device, use_arc=False,
                                      arc_epsilon=None, arc_theta=None):
    """
    BWT/FWT from saved checkpoints.
    epsilon/theta passed explicitly to avoid CFG key confusion.
    """
    n_tasks = len(tasks)
    if n_tasks < 2:
        return float("nan"), float("nan")
 
    epsilon = arc_epsilon if arc_epsilon is not None else cfg.get("atc_arc_epsilon", 0.55)
    theta   = arc_theta   if arc_theta   is not None else cfg.get("atc_arc_theta",   0.15)
 
    R = {}
    clf_tmp = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
    for task in tasks:
        tid = task["task_id"]
        if tid > 0:
            clf_tmp.grow(len(task["new_classes"]), device)
        clf_tmp.load_state_dict(clf_checkpoints[tid])
        clf_tmp.eval()
        R[tid] = {}
        for pt in tasks[:tid + 1]:
            # Fresh arc copy per prev_task to avoid retention mutations bleeding across evaluations
            fresh_arc = copy.deepcopy(clf_tmp) if use_arc else None
            res, _ = evaluate_task(
                clf_tmp, X_te, y_te, pt["all_classes"],
                task_id=tid, n_cpt=cfg["classes_per_task"],
                epsilon=epsilon, theta=theta,
                temp=cfg["arc_temp"], arc_lr=cfg["arc_lr"],
                use_arc=use_arc, device=device,
                persistent_clf_arc=fresh_arc,
            )
            R[tid][pt["task_id"]] = res["accuracy"]
 
    final_tid = n_tasks - 1
    bwt = float(np.mean([R[final_tid][i] - R[i][i] for i in range(n_tasks - 1)]))
    fwt = float(np.mean([R[i][i] - 1.0 / tasks[i]["n_classes"] for i in range(1, n_tasks)]))
    return bwt, fwt
 
 
print("compute_bwt_fwt_from_checkpoints defined")
 

compute_bwt_fwt_from_checkpoints defined


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 14 — Ablation Study  (Paper Table 4)
# 4 conditions: Baseline | Retention-only | Correction-only | Full ARC
# ─────────────────────────────────────────────────────────────────────────────
 
def run_ablation_study(tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
                        feat_dim, label_map, cfg, device,
                        dataset_name="Dataset",
                        arc_epsilon=None, arc_theta=None,
                        clf_lr_per_task=None, ewc_lambda=None):
    """
    4-condition ablation — paper Table 4 style.
    Reports ΔAB per component + OTD retention%/correction%.
    """
    print(f'\n{"="*60}\n  ABLATION — {dataset_name}\n{"="*60}')
 
    epsilon = arc_epsilon if arc_epsilon is not None else cfg.get("atc_arc_epsilon", 0.55)
    theta   = arc_theta   if arc_theta   is not None else cfg.get("atc_arc_theta",   0.15)
    lam     = ewc_lambda  if ewc_lambda  is not None else cfg.get("atc_ewc_lambda", 500.0)
 
    # ── Train shared classifier ──
    clf_base  = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
    clf_ckpts = {}
    ewc_state = None
 
    for task in tasks:
        tid = task["task_id"]
        if tid > 0:
            clf_base.grow(len(task["new_classes"]), device)
 
        if clf_lr_per_task is not None and tid < len(clf_lr_per_task):
            lr_t = clf_lr_per_task[tid]
        else:
            lr_t = cfg.get("clf_lr", 1e-3)
 
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val, task["all_classes"],
            cfg["clf_epochs"], lr_t, cfg["clf_batch"], device,
            es_patience=cfg.get("es_patience", 20),
            es_min_delta=cfg.get("es_min_delta", 1e-4),
            ewc=ewc_state, ewc_lambda=lam,
        )
        clf_ckpts[tid] = copy.deepcopy(clf_base.state_dict())
        if tid < len(tasks) - 1:
            tr_mask   = np.isin(y_tr, task["all_classes"])
            ewc_state = EWC(
                clf_base,
                DataLoader(MolDataset(X_tr[tr_mask], y_tr[tr_mask]),
                           batch_size=cfg["clf_batch"], shuffle=False),
                device, len(task["all_classes"]),
            )
 
    final_tid = len(tasks) - 1
 
    # ── Peak accuracy per task (diagonal) ──
    peak_accs = {}
    for task in tasks:
        tid   = task["task_id"]
        clf_t = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
        for t2 in tasks[:tid + 1]:
            if t2["task_id"] > 0:
                clf_t.grow(len(t2["new_classes"]), device)
        clf_t.load_state_dict(clf_ckpts[tid])
        clf_t.eval()
        res, _ = evaluate_task(
            clf_t, X_te, y_te, task["all_classes"],
            task_id=tid, n_cpt=cfg["classes_per_task"],
            epsilon=epsilon, theta=theta,
            temp=cfg["arc_temp"], arc_lr=cfg["arc_lr"],
            use_arc=False, device=device,
        )
        peak_accs[tid] = res["accuracy"]
 
    # ── 4 conditions ──
    conditions = [
        ("Baseline",         False, "full"),
        ("Retention-only",   True,  "retention"),
        ("Correction-only",  True,  "correction"),
        ("Full ARC",         True,  "full"),
    ]
    ablation_rows = []
 
    for cond_name, use_arc, arc_mode in conditions:
        print(f"\n--- {cond_name} ---")
        clf_eval = ExpandingLinearHead(feat_dim, tasks[0]["n_classes"]).to(device)
        for task in tasks:
            if task["task_id"] > 0:
                clf_eval.grow(len(task["new_classes"]), device)
        clf_eval.load_state_dict(clf_ckpts[final_tid])
        clf_eval.eval()
 
        acc_list, f1_list, auroc_list, forget_list = [], [], [], []
        total_ret = total_cor = total_cur = 0
 
        for prev_task in tasks:
            fresh_arc = copy.deepcopy(clf_eval) if use_arc else None
            res, _ = evaluate_task(
                clf_eval, X_te, y_te, prev_task["all_classes"],
                task_id=final_tid, n_cpt=cfg["classes_per_task"],
                epsilon=epsilon, theta=theta,
                temp=cfg["arc_temp"], arc_lr=cfg["arc_lr"],
                use_arc=use_arc, arc_mode=arc_mode, device=device,
                persistent_clf_arc=fresh_arc,
            )
            acc_list.append(res["accuracy"])
            f1_list.append(res["macro_f1"])
            auroc_list.append(res["auroc"])
            total_ret += res["otd_retention"]
            total_cor += res["otd_correction"]
            total_cur += res["otd_current"]
            if prev_task["task_id"] < final_tid:
                forget_list.append(peak_accs[prev_task["task_id"]] - res["accuracy"])
 
        avg_acc    = float(np.mean(acc_list))
        avg_f1     = float(np.mean(f1_list))
        forgetting = float(np.mean(forget_list)) if forget_list else 0.0
        avg_auroc  = float(np.nanmean(auroc_list))
        total_otd  = max(total_ret + total_cor + total_cur, 1)
        ret_pct    = round(100 * total_ret / total_otd, 1)
        cor_pct    = round(100 * total_cor / total_otd, 1)
 
        print(f"  AB={avg_acc:.4f}  F1={avg_f1:.4f}  Forgetting={forgetting:.4f}  AUROC={avg_auroc:.4f}")
        if use_arc:
            print(f"  OTD → retention={total_ret}({ret_pct}%)  "
                  f"correction={total_cor}({cor_pct}%)  "
                  f"current={total_cur}({100*total_cur/total_otd:.1f}%)")
 
        ablation_rows.append({
            "Dataset"    : dataset_name,
            "Condition"  : cond_name,
            "AB"         : round(avg_acc, 4),
            "Macro-F1"   : round(avg_f1, 4),
            "Forgetting" : round(forgetting, 4),
            "AUROC"      : round(avg_auroc, 4),
            "Ret%"       : ret_pct if use_arc else 0.0,
            "Cor%"       : cor_pct if use_arc else 0.0,
        })
 
    abl_df  = pd.DataFrame(ablation_rows)
    base_ab = abl_df.loc[abl_df["Condition"] == "Baseline", "AB"].values[0]
    base_f1 = abl_df.loc[abl_df["Condition"] == "Baseline", "Macro-F1"].values[0]
    abl_df["ΔAB"] = (abl_df["AB"] - base_ab).round(4)
    abl_df["ΔF1"] = (abl_df["Macro-F1"] - base_f1).round(4)
 
    print(f'\n{"─"*65}')
    print(f"  ABLATION SUMMARY — {dataset_name}  (paper Table 4 style)")
    print(f'{"─"*65}')
    print(abl_df[["Condition", "AB", "ΔAB", "Macro-F1", "ΔF1", "Forgetting", "Ret%", "Cor%"]].to_string(index=False))
    return abl_df
 
 
print("run_ablation_study defined (paper Table 4 style)")
 

run_ablation_study defined (paper Table 4 style)


In [24]:

# RUN — Hyperparameter Sweep (optional, time-consuming)
# ─────────────────────────────────────────────────────────────────────────────
 
# ── Config verification ──
print(f"ATC      -> epsilon={CFG['atc_arc_epsilon']}  theta={CFG['atc_arc_theta']}")
print(f"ATC      -> ewc_lambda={CFG['atc_ewc_lambda']}  lr_t0={CFG['atc_clf_lr_task0']}  lr_others={CFG['atc_clf_lr_others']}")
print(f"DrugBank -> epsilon={CFG['db_arc_epsilon']}   theta={CFG['db_arc_theta']}")
print(f"DrugBank -> ewc_lambda={CFG['db_ewc_lambda']}   lr_t0={CFG['db_clf_lr_task0']}   lr_t1={CFG['db_clf_lr_task1']}")
print(f"Shared   -> arc_lr={CFG['arc_lr']}  arc_temp={CFG['arc_temp']}  seed={CFG['seed']}")
 
 

ATC      -> epsilon=0.45  theta=0.2
ATC      -> ewc_lambda=500.0  lr_t0=0.0005  lr_others=0.0003
DrugBank -> epsilon=0.85   theta=0.1
DrugBank -> ewc_lambda=8000.0   lr_t0=0.0005   lr_t1=0.0003


KeyError: 'arc_lr'

In [ ]:

# RUN — ATC (14-class, 7-task CIL)
# ─────────────────────────────────────────────────────────────────────────────
 
# LR per task: task0 higher, all others lower
atc_lr_schedule = [CFG["atc_clf_lr_task0"]] + \
                  [CFG["atc_clf_lr_others"]] * (len(atc_tasks) - 1)
 
# ATC Baseline — no ARC
atc_res_base, atc_AB_base, atc_F1_base, atc_F_base, atc_AUROC_base, atc_ckpts_base = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=ATC_LABEL_MAP,
    cfg=CFG, device=DEVICE, use_arc=False,
    name="ATC 14-class Baseline",
    clf_lr_per_task=atc_lr_schedule,
    ewc_lambda=CFG["atc_ewc_lambda"],
)
print(atc_res_base.to_string(index=False))
 

In [ ]:
# ATC + ARC Run — Using Tuned Parameters
# ─────────────────────────────────────────────────────────────────────────────

print(f"\nATC ARC: epsilon={CFG['atc_arc_epsilon']}  theta={CFG['atc_arc_theta']}")

atc_res_arc, atc_AB_arc, atc_F1_arc, atc_F_arc, atc_AUROC_arc, atc_ckpt_arc = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=ATC_LABEL_MAP,
    cfg=CFG, device=DEVICE,
    use_arc=True, 
    arc_mode="full",
    name="ATC 14-class + ARC",
    arc_epsilon=CFG["atc_arc_epsilon"],
    arc_theta=CFG["atc_arc_theta"],
    arc_temp=CFG["atc_arc_temp"],  
    arc_lr=CFG["atc_arc_lr"],
    clf_lr_per_task=atc_lr_schedule,
    ewc_lambda=CFG["atc_ewc_lambda"],
)

print(atc_res_arc.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN — DrugBank
# ─────────────────────────────────────────────────────────────────────────────
 
# Extend lr schedule to cover all DrugBank tasks (fallback to task2 lr for any extra tasks)
db_lr_schedule = ([CFG["db_clf_lr_task0"], CFG["db_clf_lr_task1"], CFG["db_clf_lr_task2"]]
                  + [CFG["db_clf_lr_task2"]] * max(0, len(db_tasks) - 3))
 
# DrugBank Baseline
db_res_base, db_AB_base, db_F1_base, db_F_base, db_AUROC_base, db_ckpts_base = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, use_arc=False,
    name="DrugBank Baseline",
    clf_lr_per_task=db_lr_schedule,
    ewc_lambda=CFG["db_ewc_lambda"],
)

In [ ]:
# DrugBank + ARC
db_res_arc, db_AB_arc, db_F1_arc, db_F_arc, db_AUROC_arc, db_ckpt_arc = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE,
    use_arc=True, arc_mode="full",
    name="DrugBank + ARC",
    arc_epsilon=CFG["db_arc_epsilon"],
    arc_theta=CFG["db_arc_theta"],
    arc_temp=CFG["db_arc_temp"],
    arc_lr=CFG["db_arc_lr"],
    clf_lr_per_task=db_lr_schedule,
    ewc_lambda=CFG["db_ewc_lambda"],
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute BWT / FWT + Retention/Correction Stats
# ─────────────────────────────────────────────────────────────────────────────
 
atc_BWT_base, atc_FWT_base = compute_bwt_fwt_from_checkpoints(
    atc_tasks, atc_X_te, atc_y_te, FEAT_DIM, atc_ckpts_base,
    CFG, DEVICE, use_arc=False)
 
atc_BWT_arc, atc_FWT_arc = compute_bwt_fwt_from_checkpoints(
    atc_tasks, atc_X_te, atc_y_te, FEAT_DIM, atc_ckpt_arc,
    CFG, DEVICE, use_arc=True,
    arc_epsilon=CFG["atc_arc_epsilon"],
    arc_theta=CFG["atc_arc_theta"])
 
db_BWT_base, db_FWT_base = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_base,
    CFG, DEVICE, use_arc=False)
 
db_BWT_arc, db_FWT_arc = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpt_arc,
    CFG, DEVICE, use_arc=True,
    arc_epsilon=CFG["db_arc_epsilon"],
    arc_theta=CFG["db_arc_theta"])
 
print(f'{"─"*55}')
print(f"  BWT / FWT Results")
print(f'{"─"*55}')
print(f"  ATC      Baseline : BWT={atc_BWT_base:.4f}  FWT={atc_FWT_base:.4f}")
print(f"  ATC      ARC      : BWT={atc_BWT_arc:.4f}   FWT={atc_FWT_arc:.4f}")
print(f"  DrugBank Baseline : BWT={db_BWT_base:.4f}   FWT={db_FWT_base:.4f}")
print(f"  DrugBank ARC      : BWT={db_BWT_arc:.4f}    FWT={db_FWT_arc:.4f}")
print(f'{"─"*55}')
print(f"  delta BWT ATC      = {atc_BWT_arc - atc_BWT_base:+.4f}")
print(f"  delta BWT DrugBank = {db_BWT_arc  - db_BWT_base:+.4f}")
 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN — Ablation Studies  (Paper Table 4)
# ─────────────────────────────────────────────────────────────────────────────
 
atc_ablation_df = run_ablation_study(
    atc_tasks, atc_X_tr, atc_y_tr, atc_X_val, atc_y_val, atc_X_te, atc_y_te,
    FEAT_DIM, ATC_LABEL_MAP, CFG, DEVICE,
    dataset_name="ATC 14-class",
    arc_epsilon=CFG["atc_arc_epsilon"],
    arc_theta=CFG["atc_arc_theta"],
    clf_lr_per_task=atc_lr_schedule,
    ewc_lambda=CFG["atc_ewc_lambda"],
)
 
db_ablation_df = run_ablation_study(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val, db_X_te, db_y_te,
    FEAT_DIM, db_label_map, CFG, DEVICE,
    dataset_name="DrugBank",
    arc_epsilon=CFG["db_arc_epsilon"],
    arc_theta=CFG["db_arc_theta"],
    clf_lr_per_task=db_lr_schedule,
    ewc_lambda=CFG["db_ewc_lambda"],
)
 
combined_ablation = pd.concat([atc_ablation_df, db_ablation_df], ignore_index=True)
print("\n=== COMBINED ABLATION TABLE ===")
print(combined_ablation.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plotting — Accuracy & Forgetting + Ablation Charts  (paper style)
# ─────────────────────────────────────────────────────────────────────────────
 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("ARC vs Baseline — MolFormer CIL Results (ATC + DrugBank)",
             fontsize=14, fontweight="bold")
 
C_BASE = "#4C72B0"
C_ARC  = "#DD8452"
 
datasets = ["ATC\n(14-class, 7-task)", "DrugBank"]
ab_base  = [atc_AB_base,   db_AB_base]
ab_arc   = [atc_AB_arc,    db_AB_arc]
f_base   = [atc_F_base,    db_F_base]
f_arc    = [atc_F_arc,     db_F_arc]
 
x = np.arange(len(datasets))
w = 0.35
 
# ── Plot 1: Average Accuracy ──
ax1 = axes[0]
b1 = ax1.bar(x - w/2, ab_base, w, label="Baseline", color=C_BASE, alpha=0.85, edgecolor="white")
b2 = ax1.bar(x + w/2, ab_arc,  w, label="ARC",      color=C_ARC,  alpha=0.85, edgecolor="white")
 
for bar in b1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in b2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9, color="#994400")
 
for i in range(len(datasets)):
    delta = ab_arc[i] - ab_base[i]
    sign  = "+" if delta >= 0 else ""
    ax1.annotate(f"Δ={sign}{delta:.3f}",
                 xy=(x[i], max(ab_base[i], ab_arc[i]) + 0.025),
                 ha="center", fontsize=9,
                 color="green" if delta >= 0 else "red", fontweight="bold")
 
ax1.set_xticks(x); ax1.set_xticklabels(datasets, fontsize=11)
ax1.set_ylabel("Average Accuracy (AB)", fontsize=11)
ax1.set_title("Average Accuracy", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.set_ylim(0, min(1.0, max(ab_base + ab_arc) + 0.14))
ax1.grid(axis="y", alpha=0.3)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
 
# ── Plot 2: Forgetting ──
ax2 = axes[1]
b3 = ax2.bar(x - w/2, f_base, w, label="Baseline", color=C_BASE, alpha=0.85, edgecolor="white")
b4 = ax2.bar(x + w/2, f_arc,  w, label="ARC",      color=C_ARC,  alpha=0.85, edgecolor="white")
 
for bar in b3:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in b4:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9, color="#994400")
 
for i in range(len(datasets)):
    delta = f_arc[i] - f_base[i]
    sign  = "+" if delta >= 0 else ""
    color = "green" if delta <= 0 else "red"   # lower forgetting = better
    ax2.annotate(f"Δ={sign}{delta:.3f}",
                 xy=(x[i], max(f_base[i], f_arc[i]) + 0.010),
                 ha="center", fontsize=9, color=color, fontweight="bold")
 
ax2.set_xticks(x); ax2.set_xticklabels(datasets, fontsize=11)
ax2.set_ylabel("Forgetting (F)", fontsize=11)
ax2.set_title("Forgetting  (lower = better)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.set_ylim(0, max(f_base + f_arc) + 0.14)
ax2.grid(axis="y", alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
 
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/accuracy_forgetting_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to arc_output/accuracy_forgetting_plot.png")
 
 
# ── Ablation bar chart ──
if "atc_ablation_df" in dir() and "db_ablation_df" in dir():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Ablation Study — Retention vs Correction Contribution",
                 fontsize=13, fontweight="bold")
 
    colors     = ["#4C72B0", "#55A868", "#C44E52", "#DD8452"]
    conditions = ["Baseline", "Retention-only", "Correction-only", "Full ARC"]
 
    for ax, (abl_df, title) in zip(axes, [
        (atc_ablation_df, "ATC (14-class)"),
        (db_ablation_df,  "DrugBank"),
    ]):
        ab_vals = [abl_df.loc[abl_df["Condition"] == c, "AB"].values[0] for c in conditions]
        xi  = np.arange(len(conditions))
        bs  = ax.bar(xi, ab_vals, color=colors, alpha=0.85, edgecolor="white", width=0.55)
 
        base_val = ab_vals[0]
        for j, (bar, val) in enumerate(zip(bs, ab_vals)):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.004,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)
            if j > 0:
                delta = val - base_val
                sign  = "+" if delta >= 0 else ""
                ax.text(bar.get_x() + bar.get_width()/2, val + 0.018,
                        f"({sign}{delta:.3f})", ha="center", fontsize=8,
                        color="green" if delta >= 0 else "red")
 
        # OTD stats annotation (paper Table 5 style)
        for cond in ["Retention-only", "Correction-only", "Full ARC"]:
            row = abl_df[abl_df["Condition"] == cond]
            if len(row):
                ret_pct = row["Ret%"].values[0]
                cor_pct = row["Cor%"].values[0]
                idx = conditions.index(cond)
                ax.text(idx, 0.01, f"ret={ret_pct}%\ncor={cor_pct}%",
                        ha="center", va="bottom", fontsize=7, color="gray",
                        transform=ax.get_xaxis_transform())
 
        ax.set_xticks(xi)
        ax.set_xticklabels(conditions, fontsize=9, rotation=10)
        ax.set_ylabel("Average Accuracy (AB)")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_ylim(0, max(ab_vals) + 0.12)
        ax.grid(axis="y", alpha=0.3)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
 
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/ablation_plot.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Ablation plot saved to arc_output/ablation_plot.png")
else:
    print("Run ablation study cells first")
 


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Final Comparison Table  (paper style)
# ─────────────────────────────────────────────────────────────────────────────
 
comparison = pd.DataFrame([
    {
        "Dataset"   : "ATC (14-class)",
        "Method"    : "Baseline",
        "AB"        : round(atc_AB_base,  4),
        "Macro-F1"  : round(atc_F1_base,  4),
        "Forgetting": round(atc_F_base,   4),
        "AUROC"     : round(atc_AUROC_base, 4),
    },
    {
        "Dataset"   : "ATC (14-class)",
        "Method"    : f'ARC (ε={CFG["atc_arc_epsilon"]} θ={CFG["atc_arc_theta"]})',
        "AB"        : round(atc_AB_arc,   4),
        "Macro-F1"  : round(atc_F1_arc,   4),
        "Forgetting": round(atc_F_arc,    4),
        "AUROC"     : round(atc_AUROC_arc, 4),
    },
    {
        "Dataset"   : "DrugBank",
        "Method"    : "Baseline",
        "AB"        : round(db_AB_base,   4),
        "Macro-F1"  : round(db_F1_base,   4),
        "Forgetting": round(db_F_base,    4),
        "AUROC"     : round(db_AUROC_base, 4),
    },
    {
        "Dataset"   : "DrugBank",
        "Method"    : f'ARC (ε={CFG["db_arc_epsilon"]} θ={CFG["db_arc_theta"]})',
        "AB"        : round(db_AB_arc,    4),
        "Macro-F1"  : round(db_F1_arc,    4),
        "Forgetting": round(db_F_arc,     4),
        "AUROC"     : round(db_AUROC_arc,  4),
    },
])
 
print("=" * 75)
print("  FINAL COMPARISON TABLE — ARC CIL Pipeline")
print("=" * 75)
print(comparison.to_string(index=False))
print()
print(f"  ATC      ΔAB={atc_AB_arc-atc_AB_base:+.4f}  ΔF1={atc_F1_arc-atc_F1_base:+.4f}  "
      f"ΔF={atc_F_arc-atc_F_base:+.4f}  ΔAUROC={atc_AUROC_arc-atc_AUROC_base:+.4f}")
print(f"  DrugBank ΔAB={db_AB_arc-db_AB_base:+.4f}   ΔF1={db_F1_arc-db_F1_base:+.4f}  "
      f"ΔF={db_F_arc-db_F_base:+.4f}  ΔAUROC={db_AUROC_arc-db_AUROC_base:+.4f}")
 